In [ ]:
import pandas as pd
import numpy as np

# 1) Load data (skip first row, use second row as headers)
orders = q.cells("'fact_order_line'!A2:K18008", first_row_header=True)
products = q.cells("'dim_products'!A2:E20", first_row_header=True)
customers = q.cells("'dim_customers'!A2:D37", first_row_header=True)
fx = q.cells("'exchange_rate'!A2:D80", first_row_header=True)

# 2) Clean data
for df, cols in [(orders, ["product_id", "customer_id"]), (products, ["product_id"]), (customers, ["customer_id"])]:
    for c in cols:
        df[c] = df[c].astype(str).str.strip()
        df.loc[df[c].isin(["", "None", "nan", "NaN", "NULL", "null"]), c] = pd.NA
        df[c] = pd.to_numeric(df[c], errors="coerce")

orders = orders.dropna(subset=["product_id", "customer_id"]).copy()
products = products.dropna(subset=["product_id"]).copy()
customers = customers.dropna(subset=["customer_id"]).copy()

orders["product_id"] = orders["product_id"].astype(int)
orders["customer_id"] = orders["customer_id"].astype(int)
products["product_id"] = products["product_id"].astype(int)
customers["customer_id"] = customers["customer_id"].astype(int)

# numeric fields
for c in ["order_qty", "delivery_qty", "price_INR", "price_USD"]:
    if c in orders.columns:
        orders[c] = pd.to_numeric(orders[c], errors="coerce")
    if c in products.columns:
        products[c] = pd.to_numeric(products[c], errors="coerce")

# dates
date_cols_orders = [
    "order_placement_date",
    "agreed_delivery_date",
    "actual_delivery_date",
]
for c in date_cols_orders:
    if c in orders.columns:
        orders[c] = pd.to_datetime(orders[c], errors="coerce")

fx["Date"] = pd.to_datetime(fx["Date"], errors="coerce")
fx["USD_INR_Rate"] = pd.to_numeric(fx["INR"], errors="coerce")

# 3) Merge tables
merged = orders.merge(products, on="product_id", how="left")
merged = merged.merge(customers, on="customer_id", how="left")
merged = merged.merge(
    fx[["Date", "USD_INR_Rate"]],
    left_on="order_placement_date",
    right_on="Date",
    how="left",
)

# remove USD rows without an exchange rate
merged = merged[~((merged["currency"] == "USD") & (merged["USD_INR_Rate"].isna()))].copy()

# 4) Calculate total_amount (INR)
merged["total_amount"] = np.where(
    merged["currency"] == "USD",
    merged["price_USD"] * merged["USD_INR_Rate"] * merged["order_qty"],
    merged["price_INR"] * merged["order_qty"],
)

# 5) Clean final output
drop_cols = ["price_USD", "price_INR", "currency", "Date", "USD_INR_Rate", "INR", "Base", "Timestamp"]
merged = merged.drop(columns=[c for c in drop_cols if c in merged.columns])

keep_cols = [
    "order_id",
    "order_placement_date",
    "customer_id",
    "customer_name",
    "city",
    "product_id",
    "product_name",
    "category",
    "order_qty",
    "agreed_delivery_date",
    "actual_delivery_date",
    "delivery_qty",
    "In Full",
    "On Time",
    "On Time In Full",
    "total_amount",
]
keep_cols = [c for c in keep_cols if c in merged.columns]

merged[keep_cols].sort_values(["order_placement_date", "order_id"], na_position="last").reset_index(drop=True)